# Phase 10: Aspect Extraction & Aspect Sentiment (ABSA)
## Aspect Term Extraction (ATE) & Aspect Polarity Classification (APC)

In this notebook, we extract business aspects and opinions from customer reviews across 6 primary target categories:
1. `delivery` (Logistics)
2. `product_quality` (Product Support)
3. `payment` (Payments)
4. `refund_return` (Refunds)
5. `packaging` (Returns / Replacement)
6. `customer_support` (Customer Support)

We evaluate an aspect matching engine that parses multi-issue customer reviews into structured JSON outputs containing aspect terms, target opinion phrases, aspect sentiment, and confidence scores.

In [1]:
import os
import re
import json
import pandas as pd

data_path = "../data/processed/ecommerce_reviews.csv"
if not os.path.exists(data_path):
    data_path = "data/processed/ecommerce_reviews.csv"

df = pd.read_csv(data_path)
df.head()

,review_id,product,review,sentiment,aspects,issues_json,department,priority
0,REV-00001,laptop,"The laptop is crystal clear display, but poor ...",Neutral,"product_quality,customer_support,packaging","[{""aspect"": ""product_quality"", ""opinion"": ""cry...",Product Support,MEDIUM
1,REV-00002,smartphone,support team is polite and quick for my recent...,Positive,customer_support,"[{""aspect"": ""customer_support"", ""opinion"": ""su...",Customer Support,LOW
2,REV-00003,desk lamp,I am unhappy with the desk lamp. stopped worki...,Negative,"product_quality,refund_return","[{""aspect"": ""product_quality"", ""opinion"": ""sto...",Product Support,URGENT
3,REV-00004,laptop,I am unhappy with the laptop. courier was rude...,Negative,"packaging,payment,delivery","[{""aspect"": ""packaging"", ""opinion"": ""standard ...",Returns / Replacement,MEDIUM
4,REV-00005,laptop,I am unhappy with the laptop. charged twice fo...,Negative,"payment,delivery","[{""aspect"": ""payment"", ""opinion"": ""charged twi...",Payments,HIGH


In [2]:
# Rule/Pattern-guided ABSA Engine
ASPECT_KEYWORDS = {
    "delivery": ["delivery", "delivered", "shipping", "courier", "delayed", "late", "dispatch", "tracking"],
    "product_quality": ["quality", "broken", "screen", "defective", "working", "sound", "material", "build", "performance"],
    "payment": ["payment", "charged", "deducted", "fee", "gateway", "transaction", "upi", "credit card"],
    "refund_return": ["refund", "return", "money back", "pickup", "reimbursement"],
    "packaging": ["packaging", "box", "package", "envelope", "wrap", "cushioning"],
    "customer_support": ["support", "customer service", "representative", "agent", "email", "response", "call"]
}

POSITIVE_WORDS = ["good", "great", "excellent", "fast", "prompt", "amazing", "premium", "helpful", "polite", "secure", "quick", "perfect", "love", "satisfied", "smooth"]
NEGATIVE_WORDS = ["bad", "terrible", "horrible", "late", "delayed", "broken", "defective", "failed", "useless", "rejected", "crushed", "damaged", "disappointed", "unhappy", "disconnected"]

def extract_aspects_from_text(text):
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if s.strip()]
    extracted_issues = []
    
    for sentence in sentences:
        lower_s = sentence.lower()
        matched_aspect = None
        
        for aspect, keywords in ASPECT_KEYWORDS.items():
            if any(kw in lower_s for kw in keywords):
                matched_aspect = aspect
                break
                
        if matched_aspect:
            pos_hits = sum(1 for w in POSITIVE_WORDS if w in lower_s)
            neg_hits = sum(1 for w in NEGATIVE_WORDS if w in lower_s)
            
            if neg_hits > pos_hits:
                sentiment = "Negative"
                conf = min(0.70 + 0.10 * neg_hits, 0.99)
            elif pos_hits > neg_hits:
                sentiment = "Positive"
                conf = min(0.70 + 0.10 * pos_hits, 0.99)
            else:
                sentiment = "Neutral"
                conf = 0.80
                
            extracted_issues.append({
                "aspect": matched_aspect,
                "opinion": sentence,
                "sentiment": sentiment,
                "confidence": round(conf, 2)
            })
            
    return extracted_issues

In [3]:
sample_review = "The phone is excellent, but the delivery was very late and my refund has not arrived."
extracted = extract_aspects_from_text(sample_review)
print("Sample Input:", sample_review)
print("Extracted Issues:")
print(json.dumps(extracted, indent=2))

Sample Input: The phone is excellent, but the delivery was very late and my refund has not arrived.
Extracted Issues:
[
  {
    "aspect": "delivery",
    "opinion": "The phone is excellent, but the delivery was very late and my refund has not arrived",
    "sentiment": "Neutral",
    "confidence": 0.8
  }
]
